### Stage 3: Merge Notifications with Trade Panel and Construct Exposure Features

- Objective: Integrate Stage 1 notifications with Stage 2 trade panel data and generate interpretable exposure predictors.
- Inputs: `cleaned_data/us_notif_clean_wide.csv` (Stage 1) and `clean_out/usitc_imports_hs4_panel.csv` (Stage 2)
- Steps: (1) Merge by product/time-compatible keys; (2) Construct notification exposure features, including `exposure_w`; (3) Retain panel structure (`country`, `hs4`, `date`) for downstream labeling/modeling.
- Output: `clean_out/hs4_imports_with_notif_exposure.csv`
- This stage provides exposure variables as predictors only.

3.1. Aggregate HS6 to HS4 (monthly imports)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Project'),
    Path('/mnt/g/My Drive/Project'),
    Path(r'G:/My Drive/Project'),
    Path.cwd(),
]
BASE = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())
CLEAN = BASE / 'clean_out'

TRADE_HS4 = CLEAN / 'usitc_imports_hs4_panel.csv'

trade = pd.read_csv(TRADE_HS4, dtype={'country': str, 'hs4': str, 'hts': str})
print('Using BASE:', BASE)

Using BASE: G:\My Drive\Project


In [3]:
# Parse date
trade["date"] = pd.to_datetime(trade["date"], errors="coerce")
trade = trade.dropna(subset=["date", "country", "hs4", "import_value"]).copy()

# Ensure numeric import value
trade["import_value"] = pd.to_numeric(trade["import_value"], errors="coerce")
trade = trade.dropna(subset=["import_value"]).copy()

# Aggregate imports to HS4 x country x date 
trade_hs4 = (
    trade.groupby(["country", "hs4", "date"], as_index=False)
    .agg(import_value=("import_value", "sum"))
)

# Recreate time vars + growth at HS4 level
trade_hs4["year"] = trade_hs4["date"].dt.year
trade_hs4["month"] = trade_hs4["date"].dt.month

trade_hs4 = trade_hs4.sort_values(["country", "hs4", "date"])
trade_hs4["log_import"] = np.log(trade_hs4["import_value"] + 1.0)
trade_hs4["growth"] = trade_hs4.groupby(["country", "hs4"])["log_import"].diff()
trade_hs4["growth_3m"] = trade_hs4.groupby(["country", "hs4"])["log_import"].diff(3)

# Save HS4 panel
trade_hs4.to_csv(TRADE_HS4, index=False, encoding="utf-8")
print("Saved:", TRADE_HS4)
print("Rows:", len(trade_hs4), "| HS4:", trade_hs4["hs4"].nunique(), "| Countries:", trade_hs4["country"].nunique())

Saved: G:\My Drive\Project\clean_out\usitc_imports_hs4_panel.csv
Rows: 509310 | HS4: 964 | Countries: 8


**3.2. Map notifications (ICS5) to HS4 using weights**

In [4]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

3.2.1. Load

In [6]:
# Inputs
NOTIF = BASE / "cleaned_data" / "us_notif_ics_long.csv"
TRADE_HS4 = CLEAN / "usitc_imports_hs4_panel.csv"
MAP_FILE = BASE / "ICS_HS_mapping_table.xlsx"

# Outputs
NOTIF_HS4 = CLEAN / "notif_hs4_exposure_monthly.csv"
MERGED = CLEAN / "hs4_imports_with_notif_exposure.csv"

# Load notifications ICS5 data
notifs = pd.read_csv(NOTIF, dtype=str)

# Ensure expected columns exist
required = ["description", "title", "notified_document", "distribution_date", "ics_code"]
missing = [c for c in required if c not in notifs.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {notifs.columns.tolist()}")

# Date to year/month
notifs["distribution_date"] = pd.to_datetime(notifs["distribution_date"], errors="coerce", dayfirst=True)
notifs = notifs.dropna(subset=["distribution_date"]).copy()
notifs["year"] = notifs["distribution_date"].dt.year.astype(int)
notifs["month"] = notifs["distribution_date"].dt.month.astype(int)

# ICS5 format and validation
notifs["ics5"] = notifs["ics_code"].astype(str).str.strip()
notifs = notifs[notifs["ics5"].str.match(r"\d{2}\.\d{3}", na=False)].copy()

# Keep minimal columns for mapping
notifs = notifs[["description", "title", "notified_document", "ics5", "year", "month"]].copy()

C:\Users\DGC\AppData\Local\Temp\ipykernel_18144\2130258335.py:20: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  notifs["distribution_date"] = pd.to_datetime(notifs["distribution_date"], errors="coerce", dayfirst=True)


3.2.2. Mapping

In [7]:
# Read and clean the concordance
map_df = pd.read_excel(MAP_FILE, dtype=str)
map_df.columns = map_df.columns.str.strip().str.lower()
map_df = map_df.rename(columns={
    "hs4": "hs4",
    "ics": "ics5",
    "weight": "w"
})

# Ensure types/formatting
map_df["hs4"] = map_df["hs4"].astype(str).str.zfill(4)
map_df["ics5"] = map_df["ics5"].astype(str)
map_df["w"] = pd.to_numeric(map_df["w"], errors='coerce')
map_df = map_df.dropna(subset=["hs4", "ics5", "w"]).copy()
map_df = map_df[map_df["w"] > 0].copy()

# Normalize weights so they sum to 1 for each HS4 code
map_df["w"] = map_df["w"] / map_df.groupby("ics5")["w"].transform("sum")

In [8]:
# Map notifs x ICS5 to HS4 and aggregate exposure

notif_hs4 = notifs.merge(map_df, on="ics5", how="left")
notif_hs4 = notif_hs4.dropna(subset=["hs4"]).copy()

notif_hs4.to_csv(CLEAN / "notif_hs4_detailed.csv", index=False, encoding="utf-8")

hs4_exposure = (notif_hs4
    .groupby(["description",  "title", "notified_document", "hs4", "year", "month"], as_index=False)
    .agg(
        notif_count=("notified_document", "nunique"),
        exposure_w=("w", "sum")
    )
)
hs4_exposure.to_csv(NOTIF_HS4, index=False, encoding="utf-8")
print("Saved:", NOTIF_HS4)

Saved: G:\My Drive\Project\clean_out\notif_hs4_exposure_monthly.csv


**3.3. Merge into HS4 imports panel**

In [9]:
trade = pd.read_csv(TRADE_HS4, dtype={"country": str, "hs4": str})
trade["date"] = pd.to_datetime(trade["date"], errors="coerce")
trade = trade.dropna(subset=["date"]).copy()
trade["year"] = trade["date"].dt.year.astype(int)
trade["month"] = trade["date"].dt.month.astype(int)

merged = trade.merge(hs4_exposure, on=["hs4", "year", "month"], how="left")
merged["notif_count"] = merged["notif_count"].fillna(0).astype(int)
merged["exposure_w"] = merged["exposure_w"].fillna(0.0)

merged.to_csv(MERGED, index=False, encoding="utf-8")
print("Saved merged panel:", MERGED, "| rows:", len(merged))
print("Exposure nonzero share:", (merged["exposure_w"] > 0).mean())

Saved merged panel: G:\My Drive\Project\clean_out\hs4_imports_with_notif_exposure.csv | rows: 522855
Exposure nonzero share: 0.07248089814575742


In [10]:
# 1) Check overlap in HS4 universe
print("HS4 in trade:", merged["hs4"].nunique())
print("HS4 in exposure:", hs4_exposure["hs4"].nunique())

print("HS4 overlap:",
      len(set(merged["hs4"]) & set(hs4_exposure["hs4"])))

# 2) Check any nonzero exposure?
print("Nonzero exposure share:",
      (merged["exposure_w"] > 0).mean())
print(notifs["ics5"].head())
print(map_df["ics5"].head())



HS4 in trade: 964
HS4 in exposure: 461
HS4 overlap: 359
Nonzero exposure share: 0.07248089814575742
0    03.120
1    03.120
2    65.020
3    65.020
4    13.020
Name: ics5, dtype: object
0    65.020
1    65.020
2    11.220
3    65.040
4    65.020
Name: ics5, dtype: object


In [11]:
# notif_count should not be almost always 1
print(hs4_exposure["notif_count"].describe())
print("share notif_count==1:", (hs4_exposure["notif_count"]==1).mean())
print("top exposure months:", hs4_exposure.sort_values("exposure_w", ascending=False).head(10))


count    13554.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: notif_count, dtype: float64
share notif_count==1: 1.0
top exposure months:                                              description  \
11912  The Mine Safety and Health Administration (MSH...   
6772   Proposed rule; request for comments - The Mine...   
1988   In response to a Petition for Reconsideration ...   
13     1 August 2023 - 88 FR (Federal Register) 50056...   
12247  The U.S. Department of Energy (''DOE'') is pub...   
7802   TITLE: Energy Labeling Rule; Correction AGENCY...   
445    AGENCY: Federal Trade Commission.ACTION: Final...   
12463  The United States Department of Agriculture (U...   
12682  This document amends Federal Motor Vehicle Saf...   
8506   TITLE: Pipeline Safety: Requirement of Valve I...   

                                                   title  \
11912  Testing, Evaluation, and Approval of Electric ...   
67